# Dataset management 03: Download and compose a reviewed METASPACE selection

This tutorial turns the filter configuration exported by the METASPACE explorer into a reproducible selection, downloads the selected imzML/ibd pairs and their annotation CSVs, and composes them into one merged, catalog-backed cohort dataset.

Interactive dataset discovery is outside this notebook. Complete that workflow first in [Dataset management 02](dataset_management_02_metaspace_explorer.ipynb). This notebook uses `download` followed by `compose` — there is no combined `download-merge` command.</cell id="645ae290">

```{admonition} METASPACE API access
:class: warning

As of this writing, METASPACE does not allow this project to query or
download datasets through its public API. This notebook documents the
intended download-and-compose workflow for when API access is available. To
compose datasets that are already local, see
[Compose a cohort dataset](../../how-to/dataset-management/composing-a-cohort.md)
instead.
```

## 1. Initialize the tutorial environment

The following cell locates the repository root and makes relative paths independent of the directory from which Jupyter was started.

## 2. Resolve the reviewed filters to a selection

The explorer exports filters, not downloadable files. `query` executes those filters once and stores the accepted dataset IDs in a selection JSON. The selection is the reproducible input to download.

The example uses `workspace/configs/datasets/metaspace-mouse-liver/filter.json`, exported by [Dataset management 02](dataset_management_02_metaspace_explorer.ipynb). Replace this path when the explorer exported another configuration.

In [ ]:
%%bash
set -euo pipefail
uv run msi-datasets query \
  --workspace-path workspace \
  --source metaspace \
  --filters workspace/configs/datasets/metaspace-mouse-liver/filter.json \
  --selection workspace/configs/datasets/metaspace-mouse-liver/selection.json

In [ ]:
import json

selection_path = Path("workspace/configs/datasets/metaspace-mouse-liver/selection.json")
selection = json.loads(selection_path.read_text(encoding="utf-8"))
print("Source:", selection["source"])
print("annotation_fdr:", selection["filters"].get("annotation_fdr"))
print("Datasets:", len(selection["datasets"]))
[(item["dataset_id"], item["name"]) for item in selection["datasets"]]

In [ ]:
%%bash
set -euo pipefail
if [[ -z "${METASPACE_API_KEY:-}" ]]; then
  echo "METASPACE_API_KEY is unavailable. Source assets/scripts/datasets/metaspace_session.sh and restart Jupyter from that shell." >&2
  exit 1
fi
uv run python -m msi_dataset_manager.sources.strategies.metaspace_authentication

## 4. Download imzML/ibd pairs and annotation CSVs

`download` retrieves a complete imzML/ibd pair per selected dataset, then fetches molecular results and first-isotope ion images at the selection's `annotation_fdr` (the command below omits `--annotation-options`, so it reuses that stored value) and writes them as `annotations.csv` and `pixel_intensities.csv` beside the image. It does not merge spectra and does not write anything into a SQLite catalog — see [Download selected datasets](../../how-to/dataset-management/downloading-datasets.md).

In [ ]:
%%bash
set -euo pipefail
uv run msi-datasets download \
  --workspace-path workspace \
  --source metaspace \
  --selection workspace/configs/datasets/metaspace-mouse-liver/selection.json \ 
  --profiles path_to_csv  # Path to file with api keys (key column needed)  

## 5. Compose the cohort

`compose` imports each downloaded dataset's annotation CSVs into a new, self-contained composed catalog, merges every dataset's spectra into one image, and writes a cohort-wide molecule-occurrence index. All annotated spectra are included. `--unannotated-ratio 1.0` additionally requests one randomly sampled spectrum without a molecular link per annotated spectrum, capped by availability; these spectra are controls and are not assumed to be biological background. See [Compose a cohort dataset](../../how-to/dataset-management/composing-a-cohort.md).

In [ ]:
%%bash
set -euo pipefail
uv run msi-datasets compose \
  --workspace-path workspace \
  --cohort-id metaspace-mouse-liver \
  --source metaspace \
  --selection workspace/configs/datasets/metaspace-mouse-liver/selection.json \
  --row-width 128 \
  --unannotated-ratio 1.0 \
  --random-seed 0

In [ ]:
from msi_autoencoder_wrapper.annotations import SQLiteAnnotationReader
from msi_autoencoder_wrapper.readers.strategies.pyimzml_reader import PyImzMLReader

merged_path = Path("workspace/datasets/metaspace-mouse-liver/metaspace-mouse-liver.imzML")
assert merged_path.is_file()
assert merged_path.with_suffix(".ibd").is_file()

merged_reader = PyImzMLReader(merged_path)
annotation_reader = SQLiteAnnotationReader(
    "workspace/datasets/metaspace-mouse-liver/metaspace-mouse-liver.sqlite",
    merged_dataset_id="metaspace-mouse-liver",
)

print("Merged spectra:", merged_reader.GetNumberOfSpectra())
print("First spectrum source metadata:", annotation_reader.get_spectrum_metadata(0))
annotation_reader.get_spectrum_annotations(0)

## Result

The workflow produced `workspace/datasets/metaspace-mouse-liver/metaspace-mouse-liver.imzML`, its `.ibd` companion, and reversible molecular annotation provenance in the composed catalog `workspace/datasets/metaspace-mouse-liver/metaspace-mouse-liver.sqlite`. Detailed configuration and failure handling are documented in [Dataset management how-to guides](../../how-to/dataset-management/index.md), in particular [Download selected datasets](../../how-to/dataset-management/downloading-datasets.md), [Compose a cohort dataset](../../how-to/dataset-management/composing-a-cohort.md), and [Use the msi-datasets CLI](../../how-to/dataset-management/command-line-workflow.md).